# JFDS vs. Top-K on MovieLens 1M — Real-Data Evaluation

This notebook replaces the earlier synthetic-data experiment with the **real MovieLens 1M**
dataset (`movies.dat`, `ratings.dat`, `users.dat`) and two real base recommenders — **SVD**
(explicit-feedback matrix factorization) and **BPR** (implicit pairwise ranking) — so that JFDS
and Top-K are reranking genuine predicted relevance scores, not synthetic ones.

**How this notebook is organised, and what you can safely edit:**

| Section | Status | Notes |
|---|---|---|
| Config / Data loading / Train-test split | 🔒 FIXED | MovieLens stays the dataset for every future run |
| SVD / BPR base models | 🔒 FIXED | the two relevance signals JFDS and Top-K both rerank |
| Top-K reranker | 🔒 FIXED | the constant baseline you are comparing against |
| **JFDS equation** | ✏️ **EDITABLE** | one clearly-marked cell — change the equation here only |
| Metrics, R1–R8 | 🔒 FIXED | dataset/equation-agnostic; re-runs unchanged after you edit JFDS |

**R1–R6** are the six analyses you specified (Pareto frontier, scaling law, regime analysis,
conservation invariant, coupling coefficient, information theory) — they characterise the
mathematical structure of the JFDS equation's Fairness/Diversity/Utility triple, comparing the
two base recommenders (SVD vs. BPR).

**R7–R8** are two additions I'd suggest, explained where they appear: R1–R6 study JFDS's internal
structure, but none of them is a statistical test of "is JFDS actually better than Top-K, or could
this be noise?" R7 (paired significance tests) and R8 (bootstrap confidence intervals) are the
minimum needed to make that claim defensible — both are cheap to compute and reuse data you
already have, so nothing else was added beyond what's needed to support the headline claim.

**Note on execution:** this notebook is delivered un-run on purpose (training SVD/BPR on 1M
ratings takes a few minutes) — run it top to bottom yourself.

In [3]:
# ==============================================================
# CONFIG  (safe to tune — nothing below depends on the exact values)
# ==============================================================
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit
from itertools import combinations
from IPython.display import display, Markdown

DATA_DIR = Path("../dataset/movie_lens")  # folder containing movies.dat, ratings.dat, users.dat
RANDOM_SEED        = 42
TEST_FRACTION      = 0.20     # last 20% of each user's (timestamp-sorted) ratings held out
N_FACTORS          = 20       # latent factors for SVD and BPR
SVD_EPOCHS         = 12
BPR_EPOCHS         = 12
LEARNING_RATE      = 0.01
REG                = 0.02
BATCH_SIZE         = 8192
POOL_SIZE          = 50       # candidate pool size per user (retrieval stage)
K                  = 10       # final recommendation list size
LAMBDA_F_DEFAULT   = 0.25     # JFDS fallback fairness weight (overridden by grid search)
LAMBDA_D_DEFAULT   = 0.25     # JFDS fallback diversity weight (overridden by grid search)
ANALYSIS_SAMPLE_N  = 1500     # points sampled PER base model for the R1-R6 structural analyses

RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams['figure.dpi'] = 110
C_TOPK, C_JFDS, C_SVD, C_BPR = '#3498DB', '#E74C3C', '#9B59B6', '#16A085'
print('Config loaded.')


Config loaded.


---
## 1. Load MovieLens 1M  🔒 FIXED

Standard `::`-separated MovieLens 1M format, Latin-1 encoded. This section, and the dataset it
loads, do not change when the JFDS equation changes.

In [4]:
movies_df = pd.read_csv(DATA_DIR / 'movies.dat', sep='::', engine='python',
                        encoding='latin-1', header=None,
                        names=['MovieID', 'Title', 'Genres'])

users_df = pd.read_csv(DATA_DIR / 'users.dat', sep='::', engine='python',
                       encoding='latin-1', header=None,
                       names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip'])

ratings_df = pd.read_csv(DATA_DIR / 'ratings.dat', sep='::', engine='python',
                         encoding='latin-1', header=None,
                         names=['UserID', 'MovieID', 'Rating', 'Timestamp'])

print(f"movies : {len(movies_df):,}   users : {len(users_df):,}   ratings : {len(ratings_df):,}")
movies_df.head()

movies : 3,883   users : 6,040   ratings : 1,000,209


,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
# ------------------------------------------------------------
# Contiguous ID remapping + genre vectors  🔒 FIXED
# ------------------------------------------------------------
unique_users  = np.sort(ratings_df['UserID'].unique())
unique_movies = np.sort(movies_df['MovieID'].unique())
user2idx  = {u: i for i, u in enumerate(unique_users)}
movie2idx = {m: i for i, m in enumerate(unique_movies)}
N_USERS, N_ITEMS = len(unique_users), len(unique_movies)

ratings_df['u_idx'] = ratings_df['UserID'].map(user2idx)
ratings_df['i_idx'] = ratings_df['MovieID'].map(movie2idx)
ratings_df = ratings_df.dropna(subset=['i_idx']).copy()      # defensive: drop any unmapped movie ids
ratings_df['i_idx'] = ratings_df['i_idx'].astype(int)

movies_df = movies_df.set_index('MovieID').loc[unique_movies].reset_index()
all_genres = sorted({g for gl in movies_df['Genres'] for g in gl.split('|')})
N_GENRES = len(all_genres)
genre2idx = {g: i for i, g in enumerate(all_genres)}

genre_vec = np.zeros((N_ITEMS, N_GENRES))
for _, row in movies_df.iterrows():
    idx = movie2idx[row['MovieID']]
    for g in row['Genres'].split('|'):
        genre_vec[idx, genre2idx[g]] = 1.0
genre_vec = genre_vec / genre_vec.sum(axis=1, keepdims=True)   # normalise to a distribution per item

print(f"N_USERS={N_USERS}   N_ITEMS={N_ITEMS}   N_GENRES={N_GENRES}")
print('Genres:', all_genres)


N_USERS=6040   N_ITEMS=3883   N_GENRES=18
Genres: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


---
## 2. Train / Test Split  🔒 FIXED

Time-based leave-last-N-out per user: each user's ratings are sorted by timestamp and the last 20% are held out as test, exactly as a production A/B or offline eval would do (no future-leaks-into-past).

In [6]:
# ------------------------------------------------------------
# Train / Test split  🔒 FIXED
# ------------------------------------------------------------
ratings_df = ratings_df.sort_values(['UserID', 'Timestamp']).reset_index(drop=True)
ratings_df['rank_desc']  = ratings_df.groupby('UserID').cumcount(ascending=False)
ratings_df['user_count'] = ratings_df.groupby('UserID')['UserID'].transform('count')
test_cutoff = np.maximum(1, (ratings_df['user_count'] * TEST_FRACTION).astype(int))
test_mask = ratings_df['rank_desc'] < test_cutoff

train_df = ratings_df.loc[~test_mask, ['UserID', 'MovieID', 'Rating', 'Timestamp', 'u_idx', 'i_idx']].reset_index(drop=True)
test_df  = ratings_df.loc[ test_mask, ['UserID', 'MovieID', 'Rating', 'Timestamp', 'u_idx', 'i_idx']].reset_index(drop=True)

train_seen     = train_df.groupby('u_idx')['i_idx'].apply(set).to_dict()
test_relevant  = test_df[test_df['Rating'] >= 4].groupby('u_idx')['i_idx'].apply(set).to_dict()
test_grades    = test_df.groupby('u_idx').apply(lambda g: dict(zip(g['i_idx'], g['Rating']))).to_dict()

print(f"train ratings: {len(train_df):,}   test ratings: {len(test_df):,}")
print(f"users with >=1 relevant (rating>=4) test item: {len(test_relevant):,} / {N_USERS}")


train ratings: 802,553   test ratings: 197,656
users with >=1 relevant (rating>=4) test item: 5,956 / 6040


C:\Users\dahal\AppData\Local\Temp\ipykernel_18872\3131592326.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_grades    = test_df.groupby('u_idx').apply(lambda g: dict(zip(g['i_idx'], g['Rating']))).to_dict()


In [7]:
# ------------------------------------------------------------
# Popularity tiers from TRAIN interactions only (no test leakage)  🔒 FIXED
# ------------------------------------------------------------
pop_count = np.zeros(N_ITEMS)
counts = train_df['i_idx'].value_counts()
pop_count[counts.index.values] = counts.values
pop_norm = pop_count / (pop_count.max() + 1e-12)

order_by_pop = np.argsort(-pop_count)
tier = np.empty(N_ITEMS, dtype=object)
tier[order_by_pop[:int(.2 * N_ITEMS)]]                 = 'head'
tier[order_by_pop[int(.2 * N_ITEMS):int(.5 * N_ITEMS)]] = 'mid'
tier[order_by_pop[int(.5 * N_ITEMS):]]                  = 'tail'
TIERS = ['head', 'mid', 'tail']
TARGET_SHARE = {t: 1/3 for t in TIERS}     # fairness target: equal exposure across tiers

print(pd.Series(tier).value_counts())


tail    1942
mid     1165
head     776
Name: count, dtype: int64


---
## 3. Base Recommenders: SVD and BPR

These produce the predicted-relevance signal that both Top-K and JFDS rerank. They never need to
change when the JFDS equation changes — keeping them fixed is what makes the SVD-vs-BPR
comparisons in R1 meaningful across notebook revisions.

In [8]:
# ------------------------------------------------------------
# SVD — biased matrix factorization (FunkSVD-style) on explicit ratings
# ------------------------------------------------------------
def train_svd(train_df, n_users, n_items, n_factors=N_FACTORS, n_epochs=SVD_EPOCHS,
              lr=LEARNING_RATE, reg=REG, batch_size=BATCH_SIZE, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    global_mean = train_df['Rating'].mean()
    b_u = np.zeros(n_users); b_i = np.zeros(n_items)
    P = rng.normal(0, 0.1, (n_users, n_factors))
    Q = rng.normal(0, 0.1, (n_items, n_factors))
    u_idx = train_df['u_idx'].values; i_idx = train_df['i_idx'].values
    r_val = train_df['Rating'].values.astype(float)
    n = len(train_df)

    for epoch in range(n_epochs):
        perm = rng.permutation(n)
        for start in range(0, n, batch_size):
            b = perm[start:start + batch_size]
            bu, bi, rv = u_idx[b], i_idx[b], r_val[b]
            pred = global_mean + b_u[bu] + b_i[bi] + np.sum(P[bu] * Q[bi], axis=1)
            err = rv - pred
            np.add.at(b_u, bu, lr * (err - reg * b_u[bu]))
            np.add.at(b_i, bi, lr * (err - reg * b_i[bi]))
            np.add.at(P, bu, lr * (err[:, None] * Q[bi] - reg * P[bu]))
            np.add.at(Q, bi, lr * (err[:, None] * P[bu] - reg * Q[bi]))
        pred_all = global_mean + b_u[u_idx] + b_i[i_idx] + np.sum(P[u_idx] * Q[i_idx], axis=1)
        rmse = np.sqrt(np.mean((r_val - pred_all) ** 2))
        print(f"  [SVD] epoch {epoch + 1:>2}/{n_epochs}  train RMSE={rmse:.4f}")

    return {'global_mean': global_mean, 'b_u': b_u, 'b_i': b_i, 'P': P, 'Q': Q}

def svd_score_user(model, u):
    return model['global_mean'] + model['b_u'][u] + model['b_i'] + model['Q'] @ model['P'][u]


In [9]:
print('Training SVD ...')
svd_model = train_svd(train_df, N_USERS, N_ITEMS)
print('SVD training complete.')


Training SVD ...
  [SVD] epoch  1/12  train RMSE=0.9140
  [SVD] epoch  2/12  train RMSE=0.9002
  [SVD] epoch  3/12  train RMSE=0.8923
  [SVD] epoch  4/12  train RMSE=0.8819
  [SVD] epoch  5/12  train RMSE=0.8674
  [SVD] epoch  6/12  train RMSE=0.8514
  [SVD] epoch  7/12  train RMSE=0.8358
  [SVD] epoch  8/12  train RMSE=0.8202
  [SVD] epoch  9/12  train RMSE=0.8059
  [SVD] epoch 10/12  train RMSE=0.7933
  [SVD] epoch 11/12  train RMSE=0.7819
  [SVD] epoch 12/12  train RMSE=0.7719
SVD training complete.


In [10]:
# ------------------------------------------------------------
# BPR — pairwise ranking on implicit positives (train rating >= 4)
# ------------------------------------------------------------
def train_bpr(train_df, n_users, n_items, n_factors=N_FACTORS, n_epochs=BPR_EPOCHS,
              lr=LEARNING_RATE, reg=REG, batch_size=BATCH_SIZE, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    P = rng.normal(0, 0.1, (n_users, n_factors))
    Q = rng.normal(0, 0.1, (n_items, n_factors))
    b_i = np.zeros(n_items)

    pos_df = train_df[train_df['Rating'] >= 4]
    pos_users = pos_df['u_idx'].values
    pos_items = pos_df['i_idx'].values
    n_pos = len(pos_users)

    for epoch in range(n_epochs):
        perm = rng.permutation(n_pos)
        for start in range(0, n_pos, batch_size):
            b = perm[start:start + batch_size]
            bu, bi = pos_users[b], pos_items[b]
            bj = rng.integers(0, n_items, size=len(b))   # uniform negatives; rare false negatives are an acceptable approximation at this scale
            x_uij = np.sum(P[bu] * (Q[bi] - Q[bj]), axis=1) + b_i[bi] - b_i[bj]
            sig = 1.0 / (1.0 + np.exp(x_uij))             # = sigmoid(-x_uij), the BPR gradient weight
            np.add.at(P, bu, lr * (sig[:, None] * (Q[bi] - Q[bj]) - reg * P[bu]))
            np.add.at(Q, bi, lr * (sig[:, None] * P[bu] - reg * Q[bi]))
            np.add.at(Q, bj, lr * (-sig[:, None] * P[bu] - reg * Q[bj]))
            np.add.at(b_i, bi, lr * (sig - reg * b_i[bi]))
            np.add.at(b_i, bj, lr * (-sig - reg * b_i[bj]))
        print(f"  [BPR] epoch {epoch + 1:>2}/{n_epochs} complete")

    return {'P': P, 'Q': Q, 'b_i': b_i}

def bpr_score_user(model, u):
    return model['Q'] @ model['P'][u] + model['b_i']


In [11]:
print('Training BPR ...')
bpr_model = train_bpr(train_df, N_USERS, N_ITEMS)
print('BPR training complete.')


Training BPR ...
  [BPR] epoch  1/12 complete
  [BPR] epoch  2/12 complete
  [BPR] epoch  3/12 complete
  [BPR] epoch  4/12 complete
  [BPR] epoch  5/12 complete
  [BPR] epoch  6/12 complete
  [BPR] epoch  7/12 complete
  [BPR] epoch  8/12 complete
  [BPR] epoch  9/12 complete
  [BPR] epoch 10/12 complete
  [BPR] epoch 11/12 complete
  [BPR] epoch 12/12 complete
BPR training complete.


In [12]:
# ------------------------------------------------------------
# Candidate pool per user (retrieval stage)  🔒 FIXED
# ------------------------------------------------------------
def build_candidates(score_fn, model, seen_dict, pool_size=POOL_SIZE):
    pools = {}
    for u in range(N_USERS):
        scores = score_fn(model, u)
        seen = seen_dict.get(u, set())
        order = np.argsort(-scores)
        cands = []
        for i in order:
            if i not in seen:
                cands.append(i)
                if len(cands) == pool_size:
                    break
        pools[u] = (np.array(cands), scores[np.array(cands)])
    return pools

print('Building candidate pools ...')
svd_pools = build_candidates(svd_score_user, svd_model, train_seen)
bpr_pools = build_candidates(bpr_score_user, bpr_model, train_seen)
print(f"SVD pools: {len(svd_pools)}   BPR pools: {len(bpr_pools)}")


Building candidate pools ...
SVD pools: 6040   BPR pools: 6040


---
## 4. Rerankers

`greedy_rerank` is a generic engine: it doesn't know anything about JFDS specifically, it just
calls whatever `score_fn` you give it at every step. `topk_rerank` is the fixed baseline. Both are
🔒 FIXED — only the JFDS-specific score function in the next section should change.

In [13]:
# ------------------------------------------------------------
# Generic greedy reranking engine + Top-K baseline  🔒 FIXED
# ------------------------------------------------------------
def greedy_rerank(cand_ids, cand_rel, score_fn, k=K, **params):
    rel_norm = (cand_rel - cand_rel.min()) / (cand_rel.max() - cand_rel.min() + 1e-12)
    rel_map = dict(zip(cand_ids, rel_norm))
    remaining = list(cand_ids)
    selected, tier_counts = [], {t: 0 for t in TIERS}

    for step in range(min(k, len(cand_ids))):
        best_score, best_item = -np.inf, None
        for c in remaining:
            s = score_fn(c, rel_map[c], selected, tier_counts, step, **params)
            if s > best_score:
                best_score, best_item = s, c
        selected.append(best_item)
        tier_counts[tier[best_item]] += 1
        remaining.remove(best_item)
    return selected

def topk_rerank(cand_ids, cand_rel, k=K):
    order = np.argsort(-cand_rel)
    return list(np.array(cand_ids)[order[:k]])


### JFDS EQUATION 

`fair_boost`, `diversity_term`, and `jfds_score` are the entire equation. Everything above
(data, SVD, BPR, Top-K, the greedy engine) and everything below (metrics, R1–R8) is
equation-agnostic and will re-run unchanged no matter what you put here.

In [14]:
# ==============================================================
# JFDS equation — EDIT THIS CELL
# ==============================================================
def cosine_sim(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def fair_boost(cand_idx, tier_counts, n_selected):
    t = tier[cand_idx]
    current_share = tier_counts[t] / max(1, n_selected)
    gap = max(0.0, TARGET_SHARE[t] - current_share)
    return (gap ** 2) / (TARGET_SHARE[t] ** 2)

def diversity_term(cand_idx, selected_idx):
    if not selected_idx:
        return 1.0
    sims = [cosine_sim(genre_vec[cand_idx], genre_vec[j]) for j in selected_idx]
    return 1.0 - np.mean(sims)

def jfds_score(cand_idx, rel_value, selected, tier_counts, step,
               lam_f=LAMBDA_F_DEFAULT, lam_d=LAMBDA_D_DEFAULT):
    lam_u = 1 - lam_f - lam_d
    return (lam_u * rel_value
            + lam_f * fair_boost(cand_idx, tier_counts, step)
            + lam_d * diversity_term(cand_idx, selected))

def jfds_rerank(cand_ids, cand_rel, lam_f=LAMBDA_F_DEFAULT, lam_d=LAMBDA_D_DEFAULT, k=K):
    return greedy_rerank(cand_ids, cand_rel, jfds_score, k=k, lam_f=lam_f, lam_d=lam_d)


---
## 4b. Lambda Grid Search (Greedy Selection on Validation Split)

Before running the full reranking and tests, we find the **best (λ_f, λ_d) pair** for each base
model using a small held-out **validation split** (carved from the training set — test set is never
touched here).

**Protocol:**
1. Take the last 10% of each user's *training* interactions as a validation set.
2. For each `(λ_f, λ_d)` on a grid, run JFDS reranking on the validation candidates and score by a
   composite objective: `NDCG@K − β_f·|F_gap| − β_d·|D_gap|` where `F_gap`/`D_gap` penalise how
   far fairness/diversity are from their targets. You can tune `β_f` and `β_d` in CONFIG.
3. The pair with the highest mean composite score across users is selected for each base model.
4. Those best lambdas are stored in `best_lambdas` and used in the reranking cell below.

The grid is coarse by default (step 0.1) — tighten `LAMBDA_STEP` for a finer search at the cost of
more compute.

In [15]:
# --------------------------------------------------------------
# Lambda grid-search hyper-params  (tune here)
# --------------------------------------------------------------
LAMBDA_STEP     = 0.10   # grid resolution; 0.05 gives finer results but ~4x slower
VAL_FRACTION    = 0.10   # fraction of each user's TRAIN interactions used as validation
BETA_F          = 0.5    # penalty weight on |fairness gap| in composite objective
BETA_D          = 0.5    # penalty weight on |diversity gap| in composite objective
# target diversity: mean ILD of Top-K lists (a neutral reference point)
# computed below after val split is built
print('Lambda grid-search config loaded.')


Lambda grid-search config loaded.


In [16]:
# --------------------------------------------------------------
# Step 1: carve a validation split from training data
# --------------------------------------------------------------
train_df_sorted = train_df.sort_values(['UserID', 'Timestamp']).reset_index(drop=True)
train_df_sorted['rank_desc_tr'] = train_df_sorted.groupby('UserID').cumcount(ascending=False)
train_df_sorted['user_count_tr'] = train_df_sorted.groupby('UserID')['UserID'].transform('count')
val_cutoff = np.maximum(1, (train_df_sorted['user_count_tr'] * VAL_FRACTION).astype(int))
val_mask_tr = train_df_sorted['rank_desc_tr'] < val_cutoff

val_tr_df   = train_df_sorted.loc[ val_mask_tr].copy()
pure_tr_df  = train_df_sorted.loc[~val_mask_tr].copy()

val_relevant_tr  = val_tr_df[val_tr_df['Rating'] >= 4].groupby('u_idx')['i_idx'].apply(set).to_dict()
val_grades_tr    = val_tr_df.groupby('u_idx').apply(lambda g: dict(zip(g['i_idx'], g['Rating']))).to_dict()
pure_seen_tr     = pure_tr_df.groupby('u_idx')['i_idx'].apply(set).to_dict()

# Use the FULL training pools as candidates but mask out val items from seen (so val items can appear)
# Actually: we score candidates from the original train pools (items not in pure_tr seen)
def build_val_candidates(score_fn, model, pure_seen_dict, pool_size=POOL_SIZE):
    pools = {}
    for u in range(N_USERS):
        scores = score_fn(model, u)
        seen   = pure_seen_dict.get(u, set())
        order  = np.argsort(-scores)
        cands  = []
        for i in order:
            if i not in seen:
                cands.append(i)
                if len(cands) == pool_size:
                    break
        if cands:
            pools[u] = (np.array(cands), scores[np.array(cands)])
    return pools

print('Building validation candidate pools ...')
svd_val_pools = build_val_candidates(svd_score_user, svd_model, pure_seen_tr)
bpr_val_pools = build_val_candidates(bpr_score_user, bpr_model, pure_seen_tr)
print(f'  SVD val pools: {len(svd_val_pools)}   BPR val pools: {len(bpr_val_pools)}')

# --------------------------------------------------------------
# Step 2: compute diversity target = mean ILD of Top-K on val
# --------------------------------------------------------------
def cosine_sim(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def ild(rec_list):
    if len(rec_list) < 2:
        return 0.0
    sims = [cosine_sim(genre_vec[i], genre_vec[j]) for i, j in combinations(rec_list, 2)]
    return 1.0 - np.mean(sims)


def mean_ild_topk(val_pools):
    ilds = []
    for u, (cids, crel) in val_pools.items():
        rec = topk_rerank(cids, crel)
        ilds.append(ild(rec))
    return float(np.mean(ilds))

TARGET_D_SVD = mean_ild_topk(svd_val_pools)
TARGET_D_BPR = mean_ild_topk(bpr_val_pools)
print(f'Diversity targets — SVD: {TARGET_D_SVD:.4f}   BPR: {TARGET_D_BPR:.4f}')


C:\Users\dahal\AppData\Local\Temp\ipykernel_18872\3781003354.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_grades_tr    = val_tr_df.groupby('u_idx').apply(lambda g: dict(zip(g['i_idx'], g['Rating']))).to_dict()


Building validation candidate pools ...
  SVD val pools: 6040   BPR val pools: 6040
Diversity targets — SVD: 0.7267   BPR: 0.6660


In [19]:
# --------------------------------------------------------------
# Step 3: Bayesian Optimization (Optuna)
# --------------------------------------------------------------
import optuna
import numpy as np
import pandas as pd

# --------------------------------------------------------------
# Metrics
# --------------------------------------------------------------
def precision_recall_ndcg(rec_list, relevant_set, grades, k=K):
    rec = rec_list[:k]

    hits = len(set(rec) & relevant_set)
    precision = hits / k
    recall = hits / max(1, len(relevant_set))

    dcg = sum(
        grades.get(item, 0) / np.log2(r + 2)
        for r, item in enumerate(rec)
    )

    ideal = sorted(grades.values(), reverse=True)[:k]

    idcg = sum(
        g / np.log2(r + 2)
        for r, g in enumerate(ideal)
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0

    return precision, recall, ndcg


def novelty_fairness(rec_list):
    return float(np.mean([1 - pop_norm[i] for i in rec_list]))


def composite_score_user(rec, relevant_set, grades, target_d):

    if not rec:
        return -np.inf

    _, _, ndcg = precision_recall_ndcg(rec, relevant_set, grades)

    d_val = ild(rec)
    f_val = novelty_fairness(rec)

    f_gap = abs(f_val - 1/3)
    d_gap = abs(d_val - target_d)

    return ndcg - BETA_F * f_gap - BETA_D * d_gap


# --------------------------------------------------------------
# Bayesian optimization
# --------------------------------------------------------------
def optimize_lambdas(val_pools, target_d, name, n_trials=50):

    users = [
        (u, cids, crel)
        for u, (cids, crel) in val_pools.items()
    ]

    history = []

    def objective(trial):

        # Enforce λ_f + λ_d <= 1
        lam_f = trial.suggest_float("lam_f", 0.0, 1.0)
        lam_d = trial.suggest_float("lam_d", 0.0, 1.0 - lam_f)

        scores = []

        for u, cids, crel in users:

            rel = val_relevant_tr.get(u, set())
            grd = val_grades_tr.get(u, {})

            rec = jfds_rerank(
                cids,
                crel,
                lam_f=lam_f,
                lam_d=lam_d
            )

            scores.append(
                composite_score_user(
                    rec,
                    rel,
                    grd,
                    target_d
                )
            )

        mean_score = float(np.mean(scores))

        history.append({
            "lam_f": lam_f,
            "lam_d": lam_d,
            "composite": mean_score
        })

        return mean_score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42)
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True
    )

    best_lf = study.best_params["lam_f"]
    best_ld = study.best_params["lam_d"]

    print(
        f"[{name}] "
        f"best λ_f={best_lf:.4f} "
        f"λ_d={best_ld:.4f} "
        f"λ_u={1-best_lf-best_ld:.4f} "
        f"composite={study.best_value:.5f}"
    )

    return (
        best_lf,
        best_ld,
        pd.DataFrame(history)
    )


# --------------------------------------------------------------
# Run optimization
# --------------------------------------------------------------
print("Running Bayesian Optimization...")

best_lf_svd, best_ld_svd, optuna_svd = optimize_lambdas(
    svd_val_pools,
    TARGET_D_SVD,
    "SVD",
    n_trials=50
)

best_lf_bpr, best_ld_bpr, optuna_bpr = optimize_lambdas(
    bpr_val_pools,
    TARGET_D_BPR,
    "BPR",
    n_trials=50
)

best_lambdas = {
    "SVD": {
        "lam_f": best_lf_svd,
        "lam_d": best_ld_svd
    },
    "BPR": {
        "lam_f": best_lf_bpr,
        "lam_d": best_ld_bpr
    }
}

print("\nBest lambdas:")
print(best_lambdas)

[I 2026-06-30 22:02:43,400] A new study created in memory with name: no-name-b35033d4-46ef-428d-90ef-cc41e438700a


Running Bayesian Optimization...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-30 22:04:47,830] Trial 0 finished with value: -0.33961447018971896 and parameters: {'lam_f': 0.3745401188473625, 'lam_d': 0.5946336570972584}. Best is trial 0 with value: -0.33961447018971896.
[I 2026-06-30 22:06:51,907] Trial 1 finished with value: -0.29018454981432495 and parameters: {'lam_f': 0.7319939418114051, 'lam_d': 0.16044410055080702}. Best is trial 1 with value: -0.29018454981432495.
[I 2026-06-30 22:08:57,103] Trial 2 finished with value: -0.19664940294010805 and parameters: {'lam_f': 0.15601864044243652, 'lam_d': 0.1316564673568783}. Best is trial 2 with value: -0.19664940294010805.
[I 2026-06-30 22:11:01,615] Trial 3 finished with value: -0.30875476181611317 and parameters: {'lam_f': 0.05808361216819946, 'lam_d': 0.815865506454398}. Best is trial 2 with value: -0.19664940294010805.
[I 2026-06-30 22:13:05,788] Trial 4 finished with value: -0.305566802126664 and parameters: {'lam_f': 0.6011150117432088, 'lam_d': 0.28243952187913146}. Best is trial 2 with value: -

KeyboardInterrupt: 

In [ ]:
# --------------------------------------------------------------
# Step 4: visualise the grid (heatmap of composite score)
# --------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (name, grid_df), best_lf, best_ld, color in zip(
        axes,
        [('SVD', grid_svd), ('BPR', grid_bpr)],
        [best_lf_svd, best_lf_bpr],
        [best_ld_svd, best_ld_bpr],
        [C_SVD, C_BPR]):
    pivot = grid_df.pivot(index='lam_d', columns='lam_f', values='composite')
    im = ax.imshow(pivot.values, origin='lower', aspect='auto',
                   extent=[lambdas.min(), lambdas.max(), lambdas.min(), lambdas.max()],
                   cmap='RdYlGn')
    ax.scatter([best_lf], [best_ld], marker='*', s=280, color='white',
               edgecolor='black', linewidth=1.2, zorder=5,
               label=f'best: λ_f={best_lf:.2f}, λ_d={best_ld:.2f}')
    ax.set_xlabel('λ_f (fairness weight)')
    ax.set_ylabel('λ_d (diversity weight)')
    ax.set_title(f'{name}: composite score grid\n(white star = selected λ)', fontsize=11)
    ax.legend(fontsize=9, loc='upper right')
    plt.colorbar(im, ax=ax, label='composite (NDCG − penalties)')

plt.suptitle('Lambda Grid Search — Greedy Selection', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('lambda_grid_search.png', dpi=110, bbox_inches='tight')
plt.show()


In [ ]:
# Uses the best (lam_f, lam_d) found by grid search above — NOT the defaults.
print('Re-ranking all users (Top-K and JFDS with best lambdas, on both base models) ...')
lists = {}
for model_name, pools in [('SVD', svd_pools), ('BPR', bpr_pools)]:
    lf = best_lambdas[model_name]['lam_f']
    ld = best_lambdas[model_name]['lam_d']
    topk_lists, jfds_lists = {}, {}
    for u in range(N_USERS):
        cand_ids, cand_rel = pools[u]
        if len(cand_ids) == 0:
            continue
        topk_lists[u] = topk_rerank(cand_ids, cand_rel)
        jfds_lists[u] = jfds_rerank(cand_ids, cand_rel, lam_f=lf, lam_d=ld)
    lists[(model_name, 'TopK')] = topk_lists
    lists[(model_name, 'JFDS')] = jfds_lists
    print(f"  {model_name}: λ_f={lf:.2f} λ_d={ld:.2f} — "
          f"{len(topk_lists)} Top-K lists, {len(jfds_lists)} JFDS lists")


---
## 5. Evaluation Metrics  🔒 FIXED

* **Precision@K / Recall@K / NDCG@K** against the real held-out test ratings (relevant = test
  rating ≥ 4; NDCG graded relevance = the actual test rating).
* **D (diversity)** — intra-list genre dissimilarity (ILD).
* **F (fairness)** — mean novelty `1 − normalized popularity` of the list's items (an
  independent fairness/long-tail proxy — deliberately *not* defined as 1−Gini, so R6 below is a
  genuine empirical question rather than circular).
* **H** — Shannon entropy of the list's genre distribution (for R6).
* **gini_exp** — Gini coefficient of the *within-list* popularity values (for R6).
* **calibration_kl** — Steck (2018) KL-divergence between the user's train-set genre profile and
  the recommended list's genre distribution.
* **U (Utility)** — defined as NDCG@10 (real accuracy, not an internal score term), used as the
  third axis in R1, R5.

In [ ]:
def precision_recall_ndcg(rec_list, relevant_set, grades, k=K):
    rec = rec_list[:k]
    hits = len(set(rec) & relevant_set)
    precision = hits / k
    recall = hits / max(1, len(relevant_set))
    dcg = sum(grades.get(item, 0) / np.log2(r + 2) for r, item in enumerate(rec))
    ideal = sorted(grades.values(), reverse=True)[:k]
    idcg = sum(g / np.log2(r + 2) for r, g in enumerate(ideal))
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return precision, recall, ndcg

def ild(rec_list):
    if len(rec_list) < 2:
        return 0.0
    sims = [cosine_sim(genre_vec[i], genre_vec[j]) for i, j in combinations(rec_list, 2)]
    return 1.0 - np.mean(sims)

def shannon_entropy(rec_list):
    dist = genre_vec[rec_list].mean(axis=0)
    dist = dist / dist.sum()
    return float(-np.sum([p * np.log2(p) for p in dist if p > 0]))

def gini(values):
    v = np.sort(np.asarray(values, dtype=float))
    n = len(v)
    if v.sum() == 0:
        return 0.0
    cum = np.cumsum(v)
    return float((n + 1 - 2 * np.sum(cum) / cum[-1]) / n)

def calibration_kl(rec_list, user_train_items, alpha=0.01):
    if len(user_train_items) == 0:
        return 0.0
    p = genre_vec[list(user_train_items)].mean(axis=0)
    p = p / p.sum()
    q = genre_vec[rec_list].mean(axis=0)
    q = (1 - alpha) * q + alpha * p
    q = q / q.sum()
    return float(np.sum(p * np.log((p + 1e-12) / (q + 1e-12))))

def novelty_fairness(rec_list):
    return float(np.mean([1 - pop_norm[i] for i in rec_list]))


In [ ]:
# ------------------------------------------------------------
# Per-user metrics for every (base_model, method) combination
# ------------------------------------------------------------
rows = []
for (model_name, method), rec_lists in lists.items():
    for u, rec in rec_lists.items():
        relevant_set = test_relevant.get(u, set())
        grades = test_grades.get(u, {})
        p, r, n_ndcg = precision_recall_ndcg(rec, relevant_set, grades)
        rows.append({
            'user': u, 'base_model': model_name, 'method': method,
            'precision': p, 'recall': r, 'ndcg': n_ndcg,
            'D': ild(rec), 'F': novelty_fairness(rec),
            'H': shannon_entropy(rec), 'gini_exp': gini(pop_count[rec]),
            'calibration_kl': calibration_kl(rec, train_seen.get(u, set())),
        })

metrics_long = pd.DataFrame(rows)
metrics_long['U'] = metrics_long['ndcg']     # Utility := NDCG@10
print(f"metrics_long: {len(metrics_long):,} rows")
metrics_long.head()


In [ ]:
# ------------------------------------------------------------
# Headline comparison table
# ------------------------------------------------------------
summary = (metrics_long
           .groupby(['base_model', 'method'])
           [['precision', 'recall', 'ndcg', 'D', 'F', 'H', 'gini_exp', 'calibration_kl']]
           .mean()
           .round(4))

sys_rows = []
for (model_name, method), rec_lists in lists.items():
    exposure = np.zeros(N_ITEMS)
    for rec in rec_lists.values():
        for i in rec:
            exposure[i] += 1
    coverage = (exposure > 0).sum() / N_ITEMS
    exp_gini = gini(exposure)
    tier_share = (pd.Series(tier[np.repeat(np.arange(N_ITEMS), exposure.astype(int))])
                  .value_counts(normalize=True).reindex(TIERS).fillna(0))
    sys_rows.append({'base_model': model_name, 'method': method,
                      'coverage': coverage, 'exposure_gini': exp_gini,
                      'head_share': tier_share['head'], 'mid_share': tier_share['mid'],
                      'tail_share': tier_share['tail']})

system_summary = pd.DataFrame(sys_rows).set_index(['base_model', 'method']).round(4)
display(summary)
display(system_summary)


---
## R1 — Pareto Frontier (SVD vs. BPR)

Which `(F, D, U)` triples are non-dominated? Computed on JFDS lists, faceted by base recommender,
with Top-K lists overlaid in grey for visual and quantitative reference.

In [ ]:
def pareto_mask(vals):
    # vals: (n, 3) array; maximize all three columns
    n = len(vals)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        if dominated[i]:
            continue
        diff = vals - vals[i]
        better_eq = np.all(diff >= -1e-12, axis=1)
        strictly = np.any(diff > 1e-9, axis=1)
        dom = better_eq & strictly
        dom[i] = False
        if dom.any():
            dominated[i] = True
    return ~dominated

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
pareto_stats = {}

for ax, model_name, color in zip(axes, ['SVD', 'BPR'], [C_SVD, C_BPR]):
    jfds_pts = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'JFDS')]
    topk_pts = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'TopK')]
    jfds_s = jfds_pts.sample(min(ANALYSIS_SAMPLE_N, len(jfds_pts)), random_state=RANDOM_SEED)
    topk_s = topk_pts.sample(min(ANALYSIS_SAMPLE_N, len(topk_pts)), random_state=RANDOM_SEED)

    vals = jfds_s[['F', 'D', 'U']].values
    mask = pareto_mask(vals)

    ax.scatter(topk_s['F'], topk_s['D'], s=14, c='lightgrey', alpha=0.5, label='Top-K (reference)')
    ax.scatter(jfds_s.loc[~mask, 'F'], jfds_s.loc[~mask, 'D'], s=16, c=color, alpha=0.35, label='JFDS (dominated)')
    ax.scatter(jfds_s.loc[mask, 'F'], jfds_s.loc[mask, 'D'], s=34, c=color, edgecolor='black',
               linewidth=0.5, label='JFDS (Pareto-optimal)')
    ax.set_xlabel('Fairness (F)'); ax.set_ylabel('Diversity (D)')
    ax.set_title(f'{model_name}: F-D Pareto Frontier', fontsize=11)
    ax.legend(fontsize=8, loc='lower right')

    r_pareto = np.corrcoef(jfds_s.loc[mask, 'F'], jfds_s.loc[mask, 'D'])[0, 1] if mask.sum() > 2 else np.nan

    topk_dominated = 0
    topk_vals = topk_s[['F', 'D', 'U']].values
    for i in range(len(topk_s)):
        diff = vals - topk_vals[i]
        better_eq = np.all(diff >= -1e-12, axis=1)
        strictly = np.any(diff > 1e-9, axis=1)
        if (better_eq & strictly).any():
            topk_dominated += 1
    pct_topk_dominated = 100 * topk_dominated / len(topk_s)

    pareto_stats[model_name] = {'n_pareto_optimal_jfds': int(mask.sum()), 'n_jfds_sampled': len(jfds_s),
                                 'pareto_pearson_r': r_pareto, 'pct_topk_dominated_by_jfds': pct_topk_dominated}

plt.tight_layout(); plt.savefig('r1_pareto_frontier.png', dpi=110, bbox_inches='tight'); plt.show()
pd.DataFrame(pareto_stats).T.round(3)


---
## R2 — Scaling Law (AIC/BIC)

Fit Linear, Quadratic, Power, Log, and Saturation (Michaelis-Menten) curves to the JFDS F→D
scatter (pooled across both base models — facet by `base_model` if you want per-model curves).
Lowest AIC wins.

In [ ]:
jfds_all = metrics_long[metrics_long.method == 'JFDS']
pool = jfds_all.sample(min(ANALYSIS_SAMPLE_N * 2, len(jfds_all)), random_state=RANDOM_SEED)
F_vals = pool['F'].values
D_vals = pool['D'].values
eps = 1e-6

def aic_bic(y, y_pred, k_params):
    n = len(y)
    rss = max(np.sum((y - y_pred) ** 2), 1e-12)
    aic = n * np.log(rss / n) + 2 * k_params
    bic = n * np.log(rss / n) + k_params * np.log(n)
    return aic, bic

models_fit = {}

c1 = np.polyfit(F_vals, D_vals, 1)
models_fit['Linear'] = {'params': c1, 'pred': np.polyval(c1, F_vals), 'k': 2,
                         'eq': f"D = {c1[1]:.4f} + {c1[0]:.4f}*F"}

c2 = np.polyfit(F_vals, D_vals, 2)
models_fit['Quadratic'] = {'params': c2, 'pred': np.polyval(c2, F_vals), 'k': 3,
                            'eq': f"D = {c2[2]:.4f} + {c2[1]:.4f}*F + {c2[0]:.4f}*F^2"}

try:
    popt, _ = curve_fit(lambda x, a, b: a * np.power(x + eps, b), F_vals, D_vals, p0=[1, 0.5], maxfev=10000)
    models_fit['Power'] = {'params': popt, 'pred': popt[0] * np.power(F_vals + eps, popt[1]), 'k': 2,
                            'eq': f"D = {popt[0]:.4f} * F^{popt[1]:.4f}"}
except RuntimeError:
    pass

try:
    popt, _ = curve_fit(lambda x, a, b: a + b * np.log(x + eps), F_vals, D_vals, p0=[0, 0.1], maxfev=10000)
    models_fit['Log'] = {'params': popt, 'pred': popt[0] + popt[1] * np.log(F_vals + eps), 'k': 2,
                          'eq': f"D = {popt[0]:.4f} + {popt[1]:.4f}*ln(F)"}
except RuntimeError:
    pass

try:
    popt, _ = curve_fit(lambda x, a, b: a * x / (b + x + eps), F_vals, D_vals, p0=[1, 0.1], maxfev=10000)
    models_fit['Saturation'] = {'params': popt, 'pred': popt[0] * F_vals / (popt[1] + F_vals + eps), 'k': 2,
                                 'eq': f"D = {popt[0]:.4f}*F / ({popt[1]:.4f}+F)"}
except RuntimeError:
    pass

rows = []
for name, m in models_fit.items():
    aic, bic = aic_bic(D_vals, m['pred'], m['k'])
    rows.append({'Model': name, 'Equation': m['eq'], 'AIC': aic, 'BIC': bic})
scaling_table = pd.DataFrame(rows).sort_values('AIC').reset_index(drop=True)
display(scaling_table)

best_name = scaling_table.iloc[0]['Model']
print(f"Best-fitting model by AIC: {best_name}  ->  {scaling_table.iloc[0]['Equation']}")

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(F_vals, D_vals, s=8, alpha=0.25, color='grey', label='data (JFDS lists)')
F_grid = np.linspace(F_vals.min(), F_vals.max(), 200)
colors_curve = plt.cm.tab10(np.linspace(0, 1, len(models_fit)))
for (name, m), col in zip(models_fit.items(), colors_curve):
    if name == 'Linear' or name == 'Quadratic':
        yg = np.polyval(m['params'], F_grid)
    elif name == 'Power':
        yg = m['params'][0] * np.power(F_grid + eps, m['params'][1])
    elif name == 'Log':
        yg = m['params'][0] + m['params'][1] * np.log(F_grid + eps)
    elif name == 'Saturation':
        yg = m['params'][0] * F_grid / (m['params'][1] + F_grid + eps)
    lw = 2.5 if name == best_name else 1.2
    ax.plot(F_grid, yg, color=col, linewidth=lw, label=f"{name}{' (best)' if name == best_name else ''}")
ax.set_xlabel('Fairness (F)'); ax.set_ylabel('Diversity (D)')
ax.set_title('R2 - Scaling Law Candidates: D as a function of F', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('r2_scaling_law.png', dpi=110, bbox_inches='tight'); plt.show()


---
## R3 — Regime Analysis

Divide the F range (from the same pooled sample) into 5 quintiles; fit a local OLS slope of D on F within each — is the relationship positive, negative, or flat in each regime?

In [ ]:
pool = pool.copy()
pool['F_quintile'] = pd.qcut(pool['F'], 5, labels=[f'Q{i + 1}' for i in range(5)], duplicates='drop')

slope_rows = []
for q, g in pool.groupby('F_quintile', observed=True):
    if len(g) < 3 or g['F'].std() < 1e-9:
        continue
    c, cov = np.polyfit(g['F'], g['D'], 1, cov=True)
    slope, slope_se = c[0], np.sqrt(cov[0, 0])
    slope_rows.append({'quintile': q, 'F_range': f"[{g['F'].min():.3f}, {g['F'].max():.3f}]",
                        'n': len(g), 'slope': slope, 'slope_se': slope_se})

regime_df = pd.DataFrame(slope_rows)
display(regime_df.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(regime_df['quintile'].astype(str), regime_df['slope'], yerr=regime_df['slope_se'] * 1.96,
       color=[C_JFDS if s > 0 else C_TOPK for s in regime_df['slope']], capsize=4)
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('Fairness (F) quintile'); ax.set_ylabel('Local OLS slope  dD/dF  (±95% CI)')
ax.set_title('R3 - Regime Analysis: does the F->D relationship change sign across F?', fontsize=12)
plt.tight_layout(); plt.savefig('r3_regime_analysis.png', dpi=110, bbox_inches='tight'); plt.show()

n_pos = int((regime_df['slope'] > 0).sum()); n_neg = int((regime_df['slope'] < 0).sum())
print(f"Positive-slope regimes: {n_pos}/{len(regime_df)}   Negative-slope regimes: {n_neg}/{len(regime_df)}")


---
## R4 — Conservation Invariant

Search a grid of α, β: which `F^α × D^β` has the lowest coefficient of variation (most nearly constant across all users)? Bootstrap gives a confidence interval on the exponents themselves.

In [ ]:
F_arr = pool['F'].values + 1e-6
D_arr = pool['D'].values + 1e-6
alphas = np.round(np.arange(-2, 2.01, 0.2), 2)
betas  = np.round(np.arange(-2, 2.01, 0.2), 2)

def cv_grid(F_arr, D_arr, alphas, betas):
    grid = np.zeros((len(alphas), len(betas)))
    for ai, a in enumerate(alphas):
        Fp = F_arr ** a
        for bi, b in enumerate(betas):
            Q = Fp * (D_arr ** b)
            m = np.mean(Q)
            grid[ai, bi] = np.std(Q) / abs(m) if abs(m) > 1e-9 else np.inf
    return grid

grid = cv_grid(F_arr, D_arr, alphas, betas)
best_idx = np.unravel_index(np.argmin(grid), grid.shape)
best_alpha, best_beta = alphas[best_idx[0]], betas[best_idx[1]]
best_cv = grid[best_idx]
print(f"Best invariant: F^{best_alpha} * D^{best_beta}   (CV = {best_cv:.4f})")

B = 200
boot_alpha, boot_beta = [], []
n = len(F_arr)
for _ in range(B):
    idx = RNG.integers(0, n, size=n)
    g = cv_grid(F_arr[idx], D_arr[idx], alphas, betas)
    bi = np.unravel_index(np.argmin(g), g.shape)
    boot_alpha.append(alphas[bi[0]]); boot_beta.append(betas[bi[1]])

alpha_ci = np.percentile(boot_alpha, [2.5, 97.5])
beta_ci  = np.percentile(boot_beta, [2.5, 97.5])
print(f"Bootstrap 95% CI:  alpha in [{alpha_ci[0]:.2f}, {alpha_ci[1]:.2f}]   beta in [{beta_ci[0]:.2f}, {beta_ci[1]:.2f}]")

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(grid.T, origin='lower', extent=[alphas.min(), alphas.max(), betas.min(), betas.max()],
               aspect='auto', cmap='viridis_r')
ax.scatter([best_alpha], [best_beta], color='red', marker='*', s=200, label=f'best (CV={best_cv:.3f})')
ax.set_xlabel(r'$\alpha$'); ax.set_ylabel(r'$\beta$')
ax.set_title(r'R4 - Coefficient of Variation of $F^\alpha D^\beta$', fontsize=12)
plt.colorbar(im, ax=ax, label='CV (lower = more conserved)')
ax.legend()
plt.tight_layout(); plt.savefig('r4_conservation_invariant.png', dpi=110, bbox_inches='tight'); plt.show()


---
## R5 — Coupling Coefficient ρ_FD

For each point, the local slope of D on F using its 50 nearest neighbours in F-space (1-D nearest neighbours = a centred window after sorting by F). Plot the local coupling coefficient against Utility.

In [ ]:
order = np.argsort(pool['F'].values)
F_sorted = pool['F'].values[order]
D_sorted = pool['D'].values[order]
U_sorted = pool['U'].values[order]
n = len(F_sorted)
half = 25   # 50 neighbours total

rho_fd = np.full(n, np.nan)
for pos in range(n):
    lo, hi = max(0, pos - half), min(n, pos + half + 1)
    Fw, Dw = F_sorted[lo:hi], D_sorted[lo:hi]
    if len(Fw) > 5 and np.var(Fw) > 1e-9:
        rho_fd[pos] = np.cov(Fw, Dw)[0, 1] / np.var(Fw)

valid = ~np.isnan(rho_fd)
r_coupling = np.corrcoef(rho_fd[valid], U_sorted[valid])[0, 1]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(U_sorted[valid], rho_fd[valid], s=10, alpha=0.35, color=C_JFDS)
c = np.polyfit(U_sorted[valid], rho_fd[valid], 1)
xg = np.linspace(U_sorted[valid].min(), U_sorted[valid].max(), 100)
ax.plot(xg, np.polyval(c, xg), color='black', lw=2, label=f'trend (Pearson r={r_coupling:.3f})')
ax.axhline(0, color='grey', ls='--', lw=1)
ax.set_xlabel('Utility (U = NDCG@10)'); ax.set_ylabel(r'Local coupling coefficient $\rho_{FD}$ (50-NN slope)')
ax.set_title('R5 - Coupling Coefficient vs. Utility', fontsize=12)
ax.legend()
plt.tight_layout(); plt.savefig('r5_coupling_coefficient.png', dpi=110, bbox_inches='tight'); plt.show()

print(f"Pearson r(rho_FD, Utility) = {r_coupling:.4f}")


---
## R6 — Information Theory

Is D proportional to the Shannon entropy of the list's genre distribution? Is F proportional to `1 − Gini` of the within-list popularity distribution? Two independent regressions.

In [ ]:
H_vals = pool['H'].values
one_minus_gini = 1 - pool['gini_exp'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

c1 = np.polyfit(H_vals, pool['D'].values, 1)
pred1 = np.polyval(c1, H_vals)
r2_1 = 1 - np.sum((pool['D'].values - pred1) ** 2) / np.sum((pool['D'].values - pool['D'].values.mean()) ** 2)
axes[0].scatter(H_vals, pool['D'], s=10, alpha=0.3, color=C_SVD)
xg = np.linspace(H_vals.min(), H_vals.max(), 100)
axes[0].plot(xg, np.polyval(c1, xg), color='black', lw=2, label=f'R²={r2_1:.3f}')
axes[0].set_xlabel('Shannon entropy H(list genre dist.)'); axes[0].set_ylabel('Diversity (D / ILD)')
axes[0].set_title('Is D proportional to entropy?', fontsize=11)
axes[0].legend()

c2 = np.polyfit(one_minus_gini, pool['F'].values, 1)
pred2 = np.polyval(c2, one_minus_gini)
r2_2 = 1 - np.sum((pool['F'].values - pred2) ** 2) / np.sum((pool['F'].values - pool['F'].values.mean()) ** 2)
axes[1].scatter(one_minus_gini, pool['F'], s=10, alpha=0.3, color=C_BPR)
xg2 = np.linspace(one_minus_gini.min(), one_minus_gini.max(), 100)
axes[1].plot(xg2, np.polyval(c2, xg2), color='black', lw=2, label=f'R²={r2_2:.3f}')
axes[1].set_xlabel('1 − Gini(popularity within list)'); axes[1].set_ylabel('Fairness (F / novelty)')
axes[1].set_title('Is F proportional to 1 − Gini of exposure?', fontsize=11)
axes[1].legend()

plt.suptitle('R6 - Information-Theoretic Checks', y=1.02, fontsize=12)
plt.tight_layout(); plt.savefig('r6_information_theory.png', dpi=110, bbox_inches='tight'); plt.show()

print(f"D ~ H(genre dist.):        R² = {r2_1:.4f}")
print(f"F ~ 1 - Gini(popularity):  R² = {r2_2:.4f}")


---
## Suggested Additions: R7–R8

R1–R6 characterise the **mathematical structure of the JFDS equation itself** — its own F/D/U
relationships, compared across SVD and BPR. None of them, on their own, is a statistical test of
"is JFDS actually better than Top-K, or could this be noise?" That's the one gap worth closing
before presenting these results as proof JFDS beats Top-K, so two additions are included below —
nothing else, since anything more (e.g. extra synthetic ablations, unrelated clustering plots)
wouldn't add evidence toward that specific claim:

* **R7 — Paired significance tests.** Wilcoxon signed-rank and paired t-test, per metric, per base
  model, on the *same users'* Top-K vs. JFDS lists — plus a simple win/loss/tie count, which is
  often more persuasive to a non-statistical audience than a p-value.
* **R8 — Bootstrap confidence intervals on the effect size.** Once you know a difference is
  significant, this tells you how big it is, with honest uncertainty, so you can report
  "JFDS improves NDCG by *X* ± *Y*" rather than just a p-value.

In [ ]:
# ------------------------------------------------------------
# R7 - Paired significance tests: JFDS vs Top-K, same users, same base model
# ------------------------------------------------------------
sig_rows = []
for model_name in ['SVD', 'BPR']:
    topk_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'TopK')].set_index('user')
    jfds_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'JFDS')].set_index('user')
    common = topk_df.index.intersection(jfds_df.index)
    topk_df, jfds_df = topk_df.loc[common], jfds_df.loc[common]

    for metric in ['precision', 'recall', 'ndcg', 'D', 'F', 'calibration_kl']:
        a, b = topk_df[metric].values, jfds_df[metric].values
        diff = b - a
        try:
            w_stat, w_p = stats.wilcoxon(a, b)
        except ValueError:
            w_stat, w_p = np.nan, np.nan
        t_stat, t_p = stats.ttest_rel(b, a)
        wins = int(np.sum(diff > 1e-9))
        losses = int(np.sum(diff < -1e-9))
        ties = len(diff) - wins - losses
        sig_rows.append({'base_model': model_name, 'metric': metric, 'mean_TopK': a.mean(), 'mean_JFDS': b.mean(),
                          'mean_delta': diff.mean(), 'wilcoxon_p': w_p, 'ttest_p': t_p,
                          'JFDS_wins': wins, 'JFDS_losses': losses, 'ties': ties, 'n_users': len(common)})

sig_table = pd.DataFrame(sig_rows).round(5)
sig_table


In [ ]:
# ------------------------------------------------------------
# R8 - Bootstrap 95% CI on mean(JFDS - Top-K) per metric, per base model
# ------------------------------------------------------------
def bootstrap_delta_ci(a, b, n_boot=2000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    n = len(a)
    diffs = b - a
    boots = np.array([rng.choice(diffs, size=n, replace=True).mean() for _ in range(n_boot)])
    return diffs.mean(), np.percentile(boots, [2.5, 97.5])

boot_rows = []
for model_name in ['SVD', 'BPR']:
    topk_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'TopK')].set_index('user')
    jfds_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'JFDS')].set_index('user')
    common = topk_df.index.intersection(jfds_df.index)
    topk_df, jfds_df = topk_df.loc[common], jfds_df.loc[common]
    for metric in ['precision', 'recall', 'ndcg', 'D', 'F', 'calibration_kl']:
        mean_delta, ci = bootstrap_delta_ci(topk_df[metric].values, jfds_df[metric].values)
        boot_rows.append({'base_model': model_name, 'metric': metric, 'mean_delta': mean_delta,
                           'ci_low': ci[0], 'ci_high': ci[1]})

boot_df = pd.DataFrame(boot_rows)
display(boot_df.round(5))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
for ax, model_name in zip(axes, ['SVD', 'BPR']):
    sub = boot_df[boot_df.base_model == model_name]
    y = np.arange(len(sub))
    colors_b = [C_JFDS if (lo > 0 or hi < 0) else 'grey' for lo, hi in zip(sub.ci_low, sub.ci_high)]
    ax.errorbar(sub['mean_delta'], y, xerr=[sub['mean_delta'] - sub['ci_low'], sub['ci_high'] - sub['mean_delta']],
                fmt='o', color='black', ecolor=colors_b, elinewidth=3, capsize=4)
    ax.axvline(0, color='grey', ls='--', lw=1)
    ax.set_yticks(y); ax.set_yticklabels(sub['metric'])
    ax.set_xlabel('mean(JFDS − Top-K)  with 95% bootstrap CI')
    ax.set_title(f'{model_name}', fontsize=11)

plt.suptitle('R8 - Effect Size with Uncertainty (colored = CI excludes zero)', y=1.03, fontsize=12)
plt.tight_layout(); plt.savefig('r8_bootstrap_ci.png', dpi=110, bbox_inches='tight'); plt.show()


---
## Conclusion

In [ ]:
n_sig_favor_jfds = int((((sig_table['wilcoxon_p'] < 0.05) &
                          (sig_table['mean_delta'] > 0) &
                          (sig_table['metric'] != 'calibration_kl')).sum()) +
                       (((sig_table['wilcoxon_p'] < 0.05) &
                         (sig_table['mean_delta'] < 0) &
                         (sig_table['metric'] == 'calibration_kl')).sum()))
n_metrics_tested = len(sig_table)

display(Markdown(f'''
Out of **{n_metrics_tested}** (metric x base-model) comparisons tested in R7, JFDS is statistically
significantly better than Top-K (Wilcoxon p<0.05, correct direction for that metric) on
**{n_sig_favor_jfds}** of them -- see `sig_table` for the exact numbers per metric.

**R1 (Pareto):** see `pareto_stats` for the percentage of Top-K's own recommendation lists that
are dominated by at least one JFDS list jointly on (Fairness, Diversity, Utility).

**R2 (Scaling law):** the best-fitting F→D curve was **{best_name}**: `{scaling_table.iloc[0]['Equation']}`.

**R4 (Conservation invariant):** the most stable combination found was
F^{best_alpha:.2f} x D^{best_beta:.2f} (CV={best_cv:.4f}); see the printed bootstrap CI on the
exponents above.

**R7/R8:** `sig_table` and `boot_df` hold the exact p-values and effect sizes per metric per base
model -- these are the numbers to cite when claiming JFDS outperforms Top-K, since they are paired,
significance-tested, and carry honest uncertainty intervals rather than single point estimates.

**To swap in a new JFDS equation:** edit only the cell marked "JFDS equation -- EDIT THIS CELL" --
`fair_boost`, `diversity_term`, and `jfds_score` are the entire equation. Every cell after it
(metrics, R1-R8) re-runs unchanged against whatever new equation you write there, on the same
MovieLens data and the same Top-K baseline.
'''))
